In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

try:
    from statsmodels.stats.power import NormalIndPower
    from statsmodels.stats.proportion import proportion_effectsize
except:
    ! pip install statsmodels
    from statsmodels.stats.power import NormalIndPower
    from statsmodels.stats.proportion import proportion_effectsize

## Helper Classes

In [ ]:
class PowerAnalysis:
    def __init__(self, str_uri, str_dirname_output):
        self.str_uri = str_uri
        self.str_dirname_output = str_dirname_output
        self.df_data = None
        self.cls_power = NormalIndPower()

    def import_data(self):
        self.df_data = pd.read_parquet(self.str_uri)
        print(f'Data loaded: {self.df_data.shape[0]:,} rows')
        return self

    def compute_required_sample_size(self, flt_baseline_rate, flt_mde, flt_alpha=0.05, flt_power=0.80):
        flt_effect_size = proportion_effectsize(flt_baseline_rate, flt_baseline_rate + flt_mde)
        int_n = self.cls_power.solve_power(
            effect_size=flt_effect_size,
            alpha=flt_alpha,
            power=flt_power,
            alternative='two-sided'
        )
        int_n = int(np.ceil(int_n))
        print(f'\n=== REQUIRED SAMPLE SIZE ===')
        print(f'Baseline rate: {flt_baseline_rate:.4f}')
        print(f'Minimum detectable effect (MDE): {flt_mde:.4f}')
        print(f'Significance level (alpha): {flt_alpha}')
        print(f'Statistical power: {flt_power}')
        print(f'Effect size (Cohen\'s h): {abs(flt_effect_size):.4f}')
        print(f'Required sample size per group: {int_n:,}')
        print(f'Total required sample size: {int_n * 2:,}')
        return int_n

    def compute_achieved_power(self, flt_baseline_rate, flt_mde, int_n_per_group, flt_alpha=0.05):
        flt_effect_size = proportion_effectsize(flt_baseline_rate, flt_baseline_rate + flt_mde)
        flt_achieved_power = self.cls_power.solve_power(
            effect_size=flt_effect_size,
            nobs1=int_n_per_group,
            alpha=flt_alpha,
            alternative='two-sided'
        )
        print(f'\n=== ACHIEVED POWER ===')
        print(f'Actual sample size per group: {int_n_per_group:,}')
        print(f'Achieved power: {flt_achieved_power:.4f}')
        return flt_achieved_power

    def plot_sample_size_vs_mde(self, flt_baseline_rate, flt_alpha=0.05, flt_power=0.80):
        list_mde = np.arange(0.005, 0.051, 0.005)
        list_n = []
        for flt_mde in list_mde:
            flt_effect_size = proportion_effectsize(flt_baseline_rate, flt_baseline_rate + flt_mde)
            int_n = int(np.ceil(self.cls_power.solve_power(
                effect_size=flt_effect_size,
                alpha=flt_alpha,
                power=flt_power,
                alternative='two-sided'
            )))
            list_n.append(int_n)
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(list_mde * 100, list_n, marker='o', color='#4C72B0', linewidth=2, markersize=8)
        ax.set_title('Required Sample Size vs. Minimum Detectable Effect', fontsize=14, y=1.02)
        ax.set_xlabel('Minimum Detectable Effect (Percentage Points)')
        ax.set_ylabel('Required Sample Size per Group')
        ax.grid(True, alpha=0.3)
        for flt_x, int_y in zip(list_mde * 100, list_n):
            ax.annotate(f'{int_y:,}', (flt_x, int_y), textcoords='offset points',
                        xytext=(0, 12), ha='center', fontsize=9)
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/01_sample_size_vs_mde.png', bbox_inches='tight', dpi=150)
        plt.show()
        return self

    def plot_power_vs_sample_size(self, flt_baseline_rate, flt_mde, flt_alpha=0.05):
        list_n = np.arange(1000, 55001, 2000)
        list_power = []
        flt_effect_size = proportion_effectsize(flt_baseline_rate, flt_baseline_rate + flt_mde)
        for int_n in list_n:
            flt_p = self.cls_power.solve_power(
                effect_size=flt_effect_size,
                nobs1=int_n,
                alpha=flt_alpha,
                alternative='two-sided'
            )
            list_power.append(flt_p)
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(list_n, list_power, marker='o', color='#4C72B0', linewidth=2, markersize=5)
        ax.axhline(y=0.80, color='red', linestyle='--', label='80% Power Threshold')
        int_n_actual = min(self.df_data.groupby('version').size())
        flt_actual_power = self.cls_power.solve_power(
            effect_size=flt_effect_size,
            nobs1=int_n_actual,
            alpha=flt_alpha,
            alternative='two-sided'
        )
        ax.axvline(x=int_n_actual, color='green', linestyle='--',
                   label=f'Actual n={int_n_actual:,} (power={flt_actual_power:.2f})')
        ax.set_title(f'Statistical Power vs. Sample Size (MDE = {flt_mde*100:.1f}pp)', fontsize=14, y=1.02)
        ax.set_xlabel('Sample Size per Group')
        ax.set_ylabel('Statistical Power')
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/02_power_vs_sample_size.png', bbox_inches='tight', dpi=150)
        plt.show()
        return self

    def plot_power_vs_alpha(self, flt_baseline_rate, flt_mde):
        list_alpha = np.arange(0.01, 0.11, 0.01)
        int_n_actual = min(self.df_data.groupby('version').size())
        flt_effect_size = proportion_effectsize(flt_baseline_rate, flt_baseline_rate + flt_mde)
        list_power = []
        for flt_a in list_alpha:
            flt_p = self.cls_power.solve_power(
                effect_size=flt_effect_size,
                nobs1=int_n_actual,
                alpha=flt_a,
                alternative='two-sided'
            )
            list_power.append(flt_p)
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(list_alpha, list_power, marker='o', color='#4C72B0', linewidth=2, markersize=8)
        ax.axhline(y=0.80, color='red', linestyle='--', label='80% Power Threshold')
        ax.axvline(x=0.05, color='green', linestyle='--', label='alpha = 0.05')
        ax.set_title('Statistical Power vs. Significance Level', fontsize=14, y=1.02)
        ax.set_xlabel('Significance Level (alpha)')
        ax.set_ylabel('Statistical Power')
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/03_power_vs_alpha.png', bbox_inches='tight', dpi=150)
        plt.show()
        return self

## Constants

In [ ]:
str_bucket = 'ab-testing-demo-repo'
str_task = 'power_analysis'
str_dirname_output = './output'
str_uri = f's3://{str_bucket}/00_data_collection/cookie_cats.parquet'
flt_alpha = 0.05
flt_power = 0.80
flt_mde = 0.01

## Output Directory

In [ ]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass
print(f'Output directory ready: {str_dirname_output}')

## Run Power Analysis

In [ ]:
cls_power = PowerAnalysis(str_uri, str_dirname_output)
cls_power.import_data()

In [ ]:
# Compute baseline retention rates
flt_baseline_ret1 = cls_power.df_data['retention_1'].mean()
flt_baseline_ret7 = cls_power.df_data['retention_7'].mean()
print(f'Baseline 1-day retention: {flt_baseline_ret1:.4f}')
print(f'Baseline 7-day retention: {flt_baseline_ret7:.4f}')

In [ ]:
# Required sample size for 7-day retention (primary metric)
print('--- 7-Day Retention ---')
int_n_required = cls_power.compute_required_sample_size(
    flt_baseline_rate=flt_baseline_ret7,
    flt_mde=flt_mde,
    flt_alpha=flt_alpha,
    flt_power=flt_power
)

In [ ]:
# Achieved power with actual sample sizes
int_n_control = (cls_power.df_data['version'] == 'gate_30').sum()
int_n_treatment = (cls_power.df_data['version'] == 'gate_40').sum()
int_n_min = min(int_n_control, int_n_treatment)
print(f'Control group (gate_30): {int_n_control:,}')
print(f'Treatment group (gate_40): {int_n_treatment:,}')

cls_power.compute_achieved_power(
    flt_baseline_rate=flt_baseline_ret7,
    flt_mde=flt_mde,
    int_n_per_group=int_n_min,
    flt_alpha=flt_alpha
)

In [ ]:
cls_power.plot_sample_size_vs_mde(flt_baseline_rate=flt_baseline_ret7)

In [ ]:
cls_power.plot_power_vs_sample_size(flt_baseline_rate=flt_baseline_ret7, flt_mde=flt_mde)

In [ ]:
cls_power.plot_power_vs_alpha(flt_baseline_rate=flt_baseline_ret7, flt_mde=flt_mde)

## Completion

In [ ]:
print('\n=== POWER ANALYSIS COMPLETE ===')
print(f'Visualizations saved to: {str_dirname_output}')